In [598]:
import os 
import sys
#os.chdir("c:\\Users\\balla\\Documents\\Code_projects\\Workforce_Prompt\\data")

In [599]:
from __future__ import annotations
import pandas as pd
import json
from typing import Optional
from pydantic import BaseModel, Field, ValidationError


In [600]:
MODEL_NAME = "Digital_Worker_Timecard"

ALLOWED_DECISIONS = {"APPROVE", "REJECT", "ESCALATE"}
ALLOWED_REASON_CODES = {
    "MISSING_DATA",
    "POLICY_EXEMPTION",
    "HUMAN_REVIEW_REQUIRED",
    "VALIDATION_FAILED",
    "OK", 
    "OVERTIME_JUSTIFICATION_REQUIRED",
    "SYSTEM_RISK",
    "OUT_OF_POLICY_HOURS",
    "MISSED_PUNCH_REVIEW"
}

In [601]:
df = pd.read_csv("timecards.csv")

# Fill the NaN values in string columns with empty strings
str_cols = ["notes", "expected_decision", "expected_reason_code"]
df[str_cols] = df[str_cols].fillna("")

df.dtypes
df.head(5)
df.isna().sum()


record_id                    0
employee_id                  0
week_start                   0
week_end                     0
reported_hours               2
approved_hours_prior_week    2
overtime_hours               2
missed_punches               0
vms_flag                     0
notes                        0
expected_decision            0
expected_reason_code         0
dtype: int64

##### Pydantic Model for Normalizing results

In [602]:
class ModelDecision(BaseModel):
    """ returns a new ModelDecision object as a normalized Object:
    employee_id converted to string and stripped
    decision and reason_code stripped and upper cased """
    employee_id:str
    decision: str = Field(...)
    reason_code: str = Field(...)
    # normalizes a string passed through this class
    def normalized(self):
        return ModelDecision(
            employee_id=str(self.employee_id).strip(),
            decision=self.decision.strip().upper(),
            reason_code=self.reason_code.strip().upper()
        )

In [603]:
def validate_json_output(raw_text):
    """
    Checks for the correct outputs by comparing model.decision to the ALLOWED_DECISIONS and ALLOWED_REASON_CODES lists
    Returns: (is_valid, parsed_model, error_message) 
    """
    try:
        data = json.loads(raw_text)
        model = ModelDecision(**data).normalized() # What did this do? Especially the ** part?
        
        if model.decision not in ALLOWED_DECISIONS:
            return False, None, "INVALID_DECISION"
        
        if model.reason_code not in ALLOWED_REASON_CODES:
            return False, None, "INVALID_REASON_CODE"
        
        return True, model, "" # If each decision and reason is in accordance with the model, return the 
    
    except json.JSONDecodeError:
        return False, None, "INVALID_JSON"
    except ValidationError:
        return False, None, "SCHEMA_FAIL"

In [604]:

print(df.columns.tolist())


['record_id', 'employee_id', 'week_start', 'week_end', 'reported_hours', 'approved_hours_prior_week', 'overtime_hours', 'missed_punches', 'vms_flag', 'notes', 'expected_decision', 'expected_reason_code']


### Prompt 1

In [605]:
def raw_model_response(vms_flag, worker_id, logged_hours, approved_hours_prior_week, notes):
    """ Returns the employee_id, decision, and reason_code for each record under each condition
    Returns in JSON format
    """
    # rule 1 VMS problem
    if (vms_flag or "").upper() == "Y":
        decision = "ESCALATE"
        reason_code = "SYSTEM_RISK"

    # rule 2 — Missing data
    elif worker_id is None or logged_hours is None or approved_hours_prior_week is None:
        decision = "REJECT"
        reason_code = "MISSING_DATA"

    # rule 3 — Check for negative hours
    elif logged_hours < 0:
        decision = "REJECT"
        reason_code = "OUT_OF_POLICY_HOURS"

    # rule 4 — Evidence of overtime
    elif logged_hours > approved_hours_prior_week:
        decision = "ESCALATE"
        reason_code = "OVERTIME_JUSTIFICATION_REQUIRED"

    # rule 5 — Otherwise OK
    else:
        decision = "APPROVE"
        reason_code = "OK"

    # creates a JSON to be returned with the worker id, decision and reason 
    return json.dumps({
        "employee_id": worker_id,
        "decision": decision,
        "reason_code": reason_code
    })

df["raw_prompt_json"] = df.apply(
    lambda r: raw_model_response(r["vms_flag"], r["employee_id"], r["reported_hours"], r["approved_hours_prior_week"], r["notes"]),
    axis=1
)


In [606]:
df["raw_decision"] = df["raw_prompt_json"].apply(lambda s: json.loads(s)["decision"])
df["raw_reason_code"] = df["raw_prompt_json"].apply(lambda s: json.loads(s)["reason_code"])
df.head()  # Check for clarity

,record_id,employee_id,week_start,week_end,reported_hours,approved_hours_prior_week,overtime_hours,missed_punches,vms_flag,notes,expected_decision,expected_reason_code,raw_prompt_json,raw_decision,raw_reason_code
0,1,C-2001,1/5/2026,1/11/2026,40.0,40.0,0.0,0,N,,APPROVE,OK,"{""employee_id"": ""C-2001"", ""decision"": ""APPROVE...",APPROVE,OK
1,2,C-2002,1/5/2026,1/11/2026,39.0,40.0,0.0,0,N,Left early one day approved by manager,APPROVE,OK,"{""employee_id"": ""C-2002"", ""decision"": ""APPROVE...",APPROVE,OK
2,3,C-2003,1/5/2026,1/11/2026,45.0,40.0,5.0,0,N,overtime approved by supervisor,APPROVE,OK,"{""employee_id"": ""C-2003"", ""decision"": ""ESCALAT...",ESCALATE,OVERTIME_JUSTIFICATION_REQUIRED
3,4,C-2004,1/5/2026,1/11/2026,46.0,40.0,6.0,0,N,,ESCALATE,OVERTIME_JUSTIFICATION_REQUIRED,"{""employee_id"": ""C-2004"", ""decision"": ""ESCALAT...",ESCALATE,OVERTIME_JUSTIFICATION_REQUIRED
4,5,C-2005,1/5/2026,1/11/2026,-2.0,38.0,-2.0,0,N,,REJECT,OUT_OF_POLICY_HOURS,"{""employee_id"": ""C-2005"", ""decision"": ""REJECT""...",REJECT,OUT_OF_POLICY_HOURS


In [607]:
# This validates the data in the json output, using the validate_json_output function

valid_list = []
reason_list = []
parsed_list = []
# 
for _, row in df.iterrows():
    ok, parsed, err = validate_json_output(row["raw_prompt_json"])
    valid_list.append(ok)
    reason_list.append(err)
    parsed_list.append(parsed)

df["raw_valid_json"] = valid_list
df["raw_validation_error"] = reason_list
df["raw_parsed_obj"] = parsed_list

df.head()

,record_id,employee_id,week_start,week_end,reported_hours,approved_hours_prior_week,overtime_hours,missed_punches,vms_flag,notes,expected_decision,expected_reason_code,raw_prompt_json,raw_decision,raw_reason_code,raw_valid_json,raw_validation_error,raw_parsed_obj
0,1,C-2001,1/5/2026,1/11/2026,40.0,40.0,0.0,0,N,,APPROVE,OK,"{""employee_id"": ""C-2001"", ""decision"": ""APPROVE...",APPROVE,OK,True,,employee_id='C-2001' decision='APPROVE' reason...
1,2,C-2002,1/5/2026,1/11/2026,39.0,40.0,0.0,0,N,Left early one day approved by manager,APPROVE,OK,"{""employee_id"": ""C-2002"", ""decision"": ""APPROVE...",APPROVE,OK,True,,employee_id='C-2002' decision='APPROVE' reason...
2,3,C-2003,1/5/2026,1/11/2026,45.0,40.0,5.0,0,N,overtime approved by supervisor,APPROVE,OK,"{""employee_id"": ""C-2003"", ""decision"": ""ESCALAT...",ESCALATE,OVERTIME_JUSTIFICATION_REQUIRED,True,,employee_id='C-2003' decision='ESCALATE' reaso...
3,4,C-2004,1/5/2026,1/11/2026,46.0,40.0,6.0,0,N,,ESCALATE,OVERTIME_JUSTIFICATION_REQUIRED,"{""employee_id"": ""C-2004"", ""decision"": ""ESCALAT...",ESCALATE,OVERTIME_JUSTIFICATION_REQUIRED,True,,employee_id='C-2004' decision='ESCALATE' reaso...
4,5,C-2005,1/5/2026,1/11/2026,-2.0,38.0,-2.0,0,N,,REJECT,OUT_OF_POLICY_HOURS,"{""employee_id"": ""C-2005"", ""decision"": ""REJECT""...",REJECT,OUT_OF_POLICY_HOURS,True,,employee_id='C-2005' decision='REJECT' reason_...


### Prompt 2

In [608]:
def new_model_response(vms_flag, worker_id, reported_hours, approved_hours_prior_week, overtime_hours, missed_punches, notes):
    """ New prompt with additional rules and clarification on decision and reason_code
    Returns the employee_id, decision, and reason_code for each record under each condition
    Returns in JSON format
    """

    notes = (notes or "").strip().lower()
    # rule 1 — mismatch between reported & approved
    if (vms_flag or "").upper() == "Y":
        decision = "ESCALATE"
        reason_code = "SYSTEM_RISK"
    # rule 2 - Check for missing data in the submitted employee data
    elif (worker_id is None or pd.isna(reported_hours) or  pd.isna(approved_hours_prior_week) 
          or pd.isna(overtime_hours) or pd.isna(missed_punches)):
        decision = "REJECT"
        reason_code = "MISSING_DATA"
    # rule 3 - Check for any negative hours being places
    elif reported_hours < 0 or overtime_hours < 0:
        decision = "REJECT"
        reason_code = "OUT_OF_POLICY_HOURS"
    # rule 4 - Overtime assignment 
    elif reported_hours > approved_hours_prior_week or reported_hours > 40:
        decision = "ESCALATE"
        reason_code = "OVERTIME_JUSTIFICATION_REQUIRED"
        # overtime over 10 hours
        if overtime_hours > 10:
            decision = "ESCALATE"
            reason_code = "OUT_OF_POLICY_HOURS"
        # mismatch of documented overtime hours
        if overtime_hours != (reported_hours - approved_hours_prior_week):
            decision = "REJECT"
            reason_code = "OUT_OF_POLICY_HOURS"
        if "approved" in notes:
            decision = "APPROVE"
            reason_code = "OK"

    # rule 5 - missing punches
    elif missed_punches > 0:
        decision = "ESCALATE"
        reason_code = "MISSED_PUNCH_REVIEW"
    
    elif "missing" in notes:
        decision = "ESCALATE"
        reason_code = "HUMAN_REVIEW_REQUIRED"
    
    else:
        decision = "APPROVE"
        reason_code = "OK"

    # creates a JSON to be returned with the worker id, decision and reason  
    return json.dumps({
        "employee_id": worker_id,
        "decision": decision,
        "reason_code": reason_code
    })

df["new_prompt_json"] = df.apply(
    lambda r: new_model_response(
        r["vms_flag"],
        r["employee_id"],
        r["reported_hours"],
        r["approved_hours_prior_week"],
        r["overtime_hours"],
        r["missed_punches"],
        r["notes"]
    ),
    axis=1
)

Add the decision and the reason code to the df

In [609]:
df["new_decision"] = df["new_prompt_json"].apply(lambda s: json.loads(s)["decision"])
df["new_reason_code"] = df["new_prompt_json"].apply(lambda s: json.loads(s)["reason_code"])
df.head()

,record_id,employee_id,week_start,week_end,reported_hours,approved_hours_prior_week,overtime_hours,missed_punches,vms_flag,notes,...,expected_reason_code,raw_prompt_json,raw_decision,raw_reason_code,raw_valid_json,raw_validation_error,raw_parsed_obj,new_prompt_json,new_decision,new_reason_code
0,1,C-2001,1/5/2026,1/11/2026,40.0,40.0,0.0,0,N,,...,OK,"{""employee_id"": ""C-2001"", ""decision"": ""APPROVE...",APPROVE,OK,True,,employee_id='C-2001' decision='APPROVE' reason...,"{""employee_id"": ""C-2001"", ""decision"": ""APPROVE...",APPROVE,OK
1,2,C-2002,1/5/2026,1/11/2026,39.0,40.0,0.0,0,N,Left early one day approved by manager,...,OK,"{""employee_id"": ""C-2002"", ""decision"": ""APPROVE...",APPROVE,OK,True,,employee_id='C-2002' decision='APPROVE' reason...,"{""employee_id"": ""C-2002"", ""decision"": ""APPROVE...",APPROVE,OK
2,3,C-2003,1/5/2026,1/11/2026,45.0,40.0,5.0,0,N,overtime approved by supervisor,...,OK,"{""employee_id"": ""C-2003"", ""decision"": ""ESCALAT...",ESCALATE,OVERTIME_JUSTIFICATION_REQUIRED,True,,employee_id='C-2003' decision='ESCALATE' reaso...,"{""employee_id"": ""C-2003"", ""decision"": ""APPROVE...",APPROVE,OK
3,4,C-2004,1/5/2026,1/11/2026,46.0,40.0,6.0,0,N,,...,OVERTIME_JUSTIFICATION_REQUIRED,"{""employee_id"": ""C-2004"", ""decision"": ""ESCALAT...",ESCALATE,OVERTIME_JUSTIFICATION_REQUIRED,True,,employee_id='C-2004' decision='ESCALATE' reaso...,"{""employee_id"": ""C-2004"", ""decision"": ""ESCALAT...",ESCALATE,OVERTIME_JUSTIFICATION_REQUIRED
4,5,C-2005,1/5/2026,1/11/2026,-2.0,38.0,-2.0,0,N,,...,OUT_OF_POLICY_HOURS,"{""employee_id"": ""C-2005"", ""decision"": ""REJECT""...",REJECT,OUT_OF_POLICY_HOURS,True,,employee_id='C-2005' decision='REJECT' reason_...,"{""employee_id"": ""C-2005"", ""decision"": ""REJECT""...",REJECT,OUT_OF_POLICY_HOURS


In [610]:
new_valid_list = []
new_reason_list = []
new_parsed_list = []


# itertuppples moves faster than iterrows. Optimized for a bigger dataset  of 100 rows
for row in df.itertuples(index=False):
    ok, parsed, err = validate_json_output(row.new_prompt_json)
    new_valid_list.append(ok)
    new_reason_list.append(err)
    new_parsed_list.append(parsed)

df['new_valid_json'] = new_valid_list
df['new_validation_error'] = new_reason_list
df['new_parsed_obj'] = new_parsed_list

In [611]:
df.columns
cols_to_print = ['record_id', 'employee_id', 'week_start', 'week_end', 'reported_hours',
       'approved_hours_prior_week', 'overtime_hours', 'missed_punches',
       'vms_flag', 'notes', 'expected_decision', 'expected_reason_code',
       'raw_decision', 'raw_reason_code',
       'new_decision', 'new_reason_code']
timecards_valid = df[cols_to_print]
timecards_valid.to_csv("timecards_valid.csv", index=False)

#### Summary Statistics

Measuring the results of prompts with analysis of summary statistics
Within this summary, this preceding function evaluates each prompt and the distribution of decisions as well as its accuracy.

In [612]:
def evaluate_prompt(df, prefix):
    """ Measures the validation count and rate, decision accuracy, reason accuracy, and JSON_parsed count for the given prompt 
    on the df parameter.
    Returns a summary with included statistics for the respective prompt
    """
    valid_json_col = f"{prefix}_valid_json"
    decision_col = f"{prefix}_decision"
    reason_col = f"{prefix}_reason_code"

    total = len(df)

    # Coverage = JSON parsed successfully I dont know wtf this is
    covered = df[valid_json_col].sum()
    coverage_rate = covered / total if total else 0.0

    # Checking to see if the decisions and reasons are in the allowed columns. If so, then it returns true
    valid_mask = (
        df[valid_json_col]
        & df[decision_col].isin(ALLOWED_DECISIONS)
        & df[reason_col].isin(ALLOWED_REASON_CODES)
    )
    # Counts how many valid JSONs there are and the proportion of valid / covered
    valid_count = valid_mask.sum()
    valid_rate = valid_count / covered if covered else 0.0

    # Count the amount of each decision in the df
    decision_rates = (df.loc[valid_mask, decision_col].value_counts(normalize=True)
          .reindex(ALLOWED_DECISIONS, fill_value=0.0))

    # Decision Accuracy
    decision_accuracy = ( (df[decision_col] == df["expected_decision"]).mean()*100)

    # Reason Accuracy
    reason_accuracy = (
    (df[reason_col] == df["expected_reason_code"]).mean()*100)

    # Joint Accuracy
    joint_accuracy = (
        (df[decision_col] == df["expected_decision"])
        & (df[reason_col] == df["expected_reason_code"])).mean()*100

    summary = {
        "model_name": MODEL_NAME, #might need to find this one or assign the prefix to it
        "prefix": prefix,
        "total_rows": int(total),
        "covered": int(covered),
        "coverage_rate": round(coverage_rate, 4)*100,
        "valid_count": int(valid_count),
        "valid_rate": round(valid_rate, 4)*100, 
        "decision_accuracy": round(decision_accuracy, 4)*100,
        "reason_accuracy": round(reason_accuracy, 4)*100,
        "joint_accuracy": round(joint_accuracy, 4)*100,
        **  {f"{d.lower()}_pct": round(decision_rates[d]*100, 2) for d in ALLOWED_DECISIONS}
    }

    return summary

In [613]:
raw_summary = evaluate_prompt(df, "raw")
new_summary = evaluate_prompt(df, "new")
print(raw_summary)
print(new_summary)

{'model_name': 'Digital_Worker_Timecard', 'prefix': 'raw', 'total_rows': 100, 'covered': 100, 'coverage_rate': 100.0, 'valid_count': 100, 'valid_rate': 100.0, 'decision_accuracy': 6900.0, 'reason_accuracy': 6600.0, 'joint_accuracy': 6600.0, 'escalate_pct': 57.0, 'reject_pct': 2.0, 'approve_pct': 41.0}
{'model_name': 'Digital_Worker_Timecard', 'prefix': 'new', 'total_rows': 100, 'covered': 100, 'coverage_rate': 100.0, 'valid_count': 100, 'valid_rate': 100.0, 'decision_accuracy': 8800.0, 'reason_accuracy': 7700.0, 'joint_accuracy': 7700.0, 'escalate_pct': 62.0, 'reject_pct': 10.0, 'approve_pct': 28.0}


Create a combined df

In [614]:
results_df = pd.DataFrame([raw_summary, new_summary])
results_df

,model_name,prefix,total_rows,covered,coverage_rate,valid_count,valid_rate,decision_accuracy,reason_accuracy,joint_accuracy,escalate_pct,reject_pct,approve_pct
0,Digital_Worker_Timecard,raw,100,100,100.0,100,100.0,6900.0,6600.0,6600.0,57.0,2.0,41.0
1,Digital_Worker_Timecard,new,100,100,100.0,100,100.0,8800.0,7700.0,7700.0,62.0,10.0,28.0


In [615]:
results_df.to_csv("timecards_summary.csv", index=False)